# Ölçüm sunucusu

Aşağıdaki tek hücre her şeyi yapar: depoyu ve veriyi kurar, ölçüm sunucusunu açar ve Claude'un göndereceği ölçümleri bekler.
İş bitince (ya da uzun süre iş gelmezse) Colab sunucusunu kendisi kapatır.

1. **Çalışma zamanı → Çalışma zamanı türünü değiştir** → GPU: **L4** (yoksa G4 ya da A100; ölçüm için L4 yeter)
2. Hücreyi çalıştır. Başta Drive izni isterse onayla.
3. Çıktıda **HAZIR** yazısını görünce gidebilirsin. Claude birkaç dakika içinde bağlanamazsa çıktıdaki adres ve anahtar satırlarını ona yapıştır.

Çıktılar Drive'da `homr_makam/tez_olcum/<tarih damgası>/` altına düşer.

In [ ]:
#@title ★ ÖLÇÜM SUNUCUSU — kur → Claude'un ölçümlerini çalıştır → kapat
# Tek başına çalışır. Sırayla:
#   1. Drive'ı bağlar        2. GPU kontrolü        3. depoyu çeker        4. bağımlılıklar
#   5. veriyi açar: eğitim hücresindeki arşivler ve sınav listeleri (v12 + el_yazmasi_v24, liste v24temiz)
#   6. modelleri Drive'dan Colab'ın diskine alır
#   7. ölçüm sunucusunu açar; adresi ve anahtarı Drive'daki homr_makam/tez_olcum/adres.txt dosyasına yazar
#   8. Claude ölçümleri bu adresten gönderir; her ölçümün çıktısı Drive'da homr_makam/tez_olcum/<tarih damgası>/
#      altına kopyalanır
#   9. Claude "bitti" deyince ya da BOSTA_KAPAT_DK dakika iş gelmeyince hücre biter
# Sunucudan yalnız bu deponun ölçüm modülleri çalıştırılabilir ve her istek anahtar ister (training/olcum/sunucu.py).
# Hücre nasıl biterse bitsin (bitti, hata verdi, durduruldu) Drive'a yazılanlar kaydedilir ve Colab sunucusu
# kapatılır; boşta kredi yemesin.
KAPAT = True  #@param {type:"boolean"}
BOSTA_KAPAT_DK = 45  #@param {type:"integer"}
VERI = 'symbtr_veri_v12.tar.gz'
EK_VERI = 'el_yazmasi_v24.tar.gz'
LISTE = 'v24temiz'
REPO_URL = 'https://github.com/mtalhabalci/aeu-omr.git'
BRANCH = 'tez-olcum'
# Colab diskine alinacak modeller (kisa ad, dosya adindaki surum izi); otekiler Drive yolundan okunabilir.
MODEL_IZLERI = (('tez', '8f6af68'), ('uygulama', '819bee2'), ('makale', '0c31606'))

from google.colab import drive
drive.mount('/content/drive', force_remount=True)  # yeni yuklenen dosyalar da gorunsun

def _sunucuyu_kapat(result=None):
    # Hucre nasil biterse bitsin (bitti, hata verdi ya da durduruldu) bir kez calisir.
    get_ipython().events.unregister('post_run_cell', _sunucuyu_kapat)
    if not KAPAT:
        return
    print('\nSunucu kapatiliyor (KAPAT=True). Sonuclar Drive da.', flush=True)
    try:
        drive.flush_and_unmount()  # Drive a yazilanlar tamamen gitsin
    except Exception as hata:
        print('Drive kaydi:', hata)
    from google.colab import runtime
    runtime.unassign()

get_ipython().events.register('post_run_cell', _sunucuyu_kapat)

import glob, json, os, re, secrets, shutil, subprocess, sys, tarfile, time, urllib.request
import torch
DRIVE = '/content/drive/MyDrive/homr_makam'
DAMGA = time.strftime('%m%d-%H%M')
OLCUM = f'{DRIVE}/tez_olcum'
KOSU = f'{OLCUM}/{DAMGA}'
CIKTI = '/content/olcum'
MODELLER = '/content/modeller'
PORT = 8765
os.makedirs(KOSU, exist_ok=True)
shutil.rmtree(CIKTI, ignore_errors=True)
os.makedirs(CIKTI)
os.makedirs(MODELLER, exist_ok=True)

def adim(yazi):
    print('\n' + '=' * 66 + '\n' + time.strftime('%H:%M  ') + yazi + '\n' + '=' * 66, flush=True)

def kos(*komut, **kw):
    return subprocess.run(list(komut), **kw)

def drive_da_bekle(yollar, ne):
    # Drive a yeni yuklenen dosya baglantida birkac dakika gec gorunebilir.
    for _ in range(30):
        if all(os.path.exists(y) for y in yollar):
            return
        print(ne, 'henuz gorunmuyor, bekleniyor...', flush=True)
        os.listdir(DRIVE)
        time.sleep(20)
    raise AssertionError(f'{ne} Drive da yok: {yollar}')

def arsiv_yolu(ad):
    eski = f'{DRIVE}/2_veri_arsivleri/{ad}'  # eski arsivler Drive da ayri klasorde
    return eski if not os.path.exists(f'{DRIVE}/{ad}') and os.path.exists(eski) else f'{DRIVE}/{ad}'

adim('1/7  GPU')
assert torch.cuda.is_available(), 'GPU YOK -> Calisma zamani > Calisma zamani turunu degistir > L4'
KART = torch.cuda.get_device_name(0)
print(KART)

adim('2/7  Depo')
os.chdir('/content')
if os.path.isdir('/content/homr'):
    shutil.rmtree('/content/homr')
kos('git', 'clone', '-q', '--depth', '1', '--branch', BRANCH, REPO_URL, '/content/homr', check=True)
os.chdir('/content/homr')
print(kos('git', 'log', '--oneline', '-1', capture_output=True, text=True).stdout.strip()[:150])

adim('3/7  Bagimliliklar')
kos(sys.executable, '-m', 'pip', 'install', '-q', 'transformers>=4.53.2,<5', 'albumentations', 'editdistance',
    'musicxml==1.4', 'x-transformers', 'onnxruntime', 'pymupdf', check=True)
if not os.path.exists('/usr/local/bin/cloudflared'):
    kos('wget', '-q', '-O', '/usr/local/bin/cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', check=True)
    os.chmod('/usr/local/bin/cloudflared', 0o755)

adim('4/7  Veri: ' + VERI + ' + ' + EK_VERI + ', liste ' + LISTE)
# Egitim hucresiyle ayni sira: arsiv, ek arsiv, sonra Drive daki listeler (sonraki oncekinin ustune yazar).
onekler = []
for arsiv_adi in (VERI, EK_VERI):
    arsiv = arsiv_yolu(arsiv_adi)
    drive_da_bekle([arsiv], arsiv_adi)
    with tarfile.open(arsiv) as t:
        t.extractall('/content/homr')
    print('acildi:', arsiv_adi, flush=True)
    onekler.append(arsiv_adi.split('_')[-1].split('.')[0] + '_')
drive_da_bekle([f'{DRIVE}/{LISTE}_index_{b}.txt' for b in ('train', 'val', 'test')], LISTE + ' listesi')
onekler.append(LISTE + '_')
for onek in onekler:
    for name in ('index.txt', 'index_train.txt', 'index_val.txt', 'index_test.txt'):
        source = f'{DRIVE}/{onek}{name}'
        if os.path.exists(source):
            shutil.copy(source, f'/content/homr/datasets/SymbTr-2.0.0/{name}')
            print('Drive dan alindi:', onek + name)

def duzeltilmis(yol):
    # Senyo etiketi duzeltilmis kopya varsa o, yoksa eskisi (egitim hucresindeki kural).
    yeni = 'datasets/SymbTr-segno/exams/' + yol[len('datasets/'):]
    return yeni if os.path.exists(yeni) else yol

SINAVLAR = [('gercek', 'datasets/SymbTr-2.0.0/index_test.txt')]
for _ad, _dosya in (('hatirlatma', 'datasets/SymbTr-courtesy/index_test.txt'),
                    ('usul', 'datasets/SymbTr-meters/index_test.txt'),
                    ('usul_font', 'datasets/SymbTr-meters/index_test_fonts.txt'),
                    ('font', 'datasets/SymbTr-fonts/index_test.txt'),
                    ('font_gorulmemis', 'datasets/SymbTr-fonts/index_test_unseen.txt'),
                    ('yipranmis', 'datasets/SymbTr-worn/index_test.txt'),
                    ('senyo_basili', 'datasets/SymbTr-segno/index_test_senyolu.txt'),
                    ('koda_basili', 'datasets/SymbTr-navmarks/index_test.txt'),
                    ('cift_isaret', 'datasets/double-signs-v2/index_test.txt'),
                    ('thm', 'datasets/repertukul-v17/index_test.txt'),
                    ('olcu_tekrari', 'datasets/measure-repeat-synth/index_test.txt'),
                    ('havuz_eb', 'datasets/havuz-erdincbal-v17/index_test.txt'),
                    ('havuz_trt', 'datasets/havuz-trt-v17/index_test.txt'),
                    ('tekrar_isareti', 'datasets/repeat-both-synth/index_test.txt'),
                    ('havuz_gorulmemis', 'datasets/havuz-gorulmemis-v19/index_test.txt')):
    if os.path.exists(duzeltilmis(_dosya)):
        SINAVLAR.append((_ad, duzeltilmis(_dosya)))
EL = 'datasets/dilbeyti-el-v2' if os.path.exists('datasets/dilbeyti-el-v2/index_test.txt') else 'datasets/dilbeyti-el'
if os.path.exists(f'{EL}/index_test.txt'):
    SINAVLAR.append(('el_yazmasi', f'{EL}/index_test.txt'))
for _dilim, _dosya in (('el_gorulmus', 'index_test_gorulmus_el.txt'),
                       ('el_gorulmemis', 'index_test_gorulmemis_el.txt'),
                       ('el_belirsiz', 'index_test_belirsiz_el.txt')):
    if os.path.exists(f'{EL}/{_dosya}'):
        SINAVLAR.append((_dilim, f'{EL}/{_dosya}'))
_sinavlar = [{'ad': ad, 'indeks': indeks, 'porte': sum(1 for _ in open(indeks))} for ad, indeks in SINAVLAR]
for s in _sinavlar:
    print('%-18s %5d porte  %s' % (s['ad'], s['porte'], s['indeks']))
_veri_klasorleri = sorted(d for d in os.listdir('datasets') if os.path.isdir(f'datasets/{d}'))

adim('5/7  Modeller')
_modeller = {}
for kisa, iz in MODEL_IZLERI:
    bulunan = glob.glob(f'{DRIVE}/pytorch_model_1-{iz}*.pth') + glob.glob(f'{DRIVE}/1_modeller/pytorch_model_1-{iz}*.pth')
    if not bulunan:
        print('YOK   ', kisa, iz)
        continue
    hedef = f'{MODELLER}/{os.path.basename(bulunan[0])}'
    shutil.copy(bulunan[0], hedef)
    _modeller[kisa] = hedef
    print('alindi', kisa, os.path.basename(hedef), flush=True)
assert 'tez' in _modeller, 'tezin modeli Drive kokunde yok'
json.dump({'sinavlar': _sinavlar, 'modeller': _modeller, 'veri_klasorleri': _veri_klasorleri, 'kart': KART,
           'drive': DRIVE, 'kosu': KOSU, 'damga': DAMGA},
          open(f'{CIKTI}/kurulum.json', 'w'), ensure_ascii=False, indent=1)
shutil.copy(f'{CIKTI}/kurulum.json', f'{KOSU}/kurulum.json')

adim('6/7  Sunucu ve adres')
kos('bash', '-c', 'pkill -f training.olcum.sunucu; pkill -f cloudflared; true')
time.sleep(2)
ANAHTAR = secrets.token_hex(16)
_slog = open('/content/sunucu.log', 'w')
srv = subprocess.Popen([sys.executable, '-m', 'training.olcum.sunucu', '--port', str(PORT), '--cikti', CIKTI,
                        '--drive', KOSU, '--kok', MODELLER, '--kok', DRIVE, '--dal', BRANCH],
                       cwd='/content/homr', env=dict(os.environ, OLCUM_ANAHTAR=ANAHTAR),
                       stdout=_slog, stderr=subprocess.STDOUT)

def yerel(yol):
    istek = urllib.request.Request(f'http://127.0.0.1:{PORT}{yol}', headers={'X-Anahtar': ANAHTAR})
    return json.load(urllib.request.urlopen(istek, timeout=10))

for _ in range(60):
    try:
        yerel('/durum?sessiz=1')
        break
    except Exception:
        if srv.poll() is not None:
            break
        time.sleep(1)
else:
    srv.kill()
if srv.poll() is not None:
    print(open('/content/sunucu.log').read()[-3000:])
    raise RuntimeError('Olcum sunucusu acilmadi')

def tunel_ac():
    # (surec, adres); adres alinamazsa (None, None)
    gunluk = open('/content/tunnel.log', 'w')
    t = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}', '--no-autoupdate'],
                         stdout=gunluk, stderr=subprocess.STDOUT)
    for _ in range(90):
        m = re.search(r'https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com', open('/content/tunnel.log').read())
        if m:
            return t, m.group(0)
        if t.poll() is not None:
            break
        time.sleep(1)
    t.kill()
    return None, None

def adres_yaz(adres):
    with open(f'{OLCUM}/adres.txt', 'w') as f:
        f.write(f'{adres}\n{ANAHTAR}\n{DAMGA}\n{int(time.time())}\n')
    print('\nADRES  :', adres)
    print('ANAHTAR:', ANAHTAR)
    print('(Drive daki homr_makam/tez_olcum/adres.txt dosyasina da yazildi. Claude birkac dakikada baglanamazsa'
          ' bu iki satiri ona yapistir.)', flush=True)

tun, adres = tunel_ac()
if not adres:
    raise RuntimeError('Tunel acilmadi; /content/tunnel.log')
adres_yaz(adres)

adim('7/7  HAZIR. Claude bekleniyor; is bitince sunucu kendini kapatir')
son_yazi = time.time()
tunel_denemesi = 0
neden = 'bilinmiyor'
try:
    while True:
        time.sleep(15)
        if srv.poll() is not None:
            neden = 'olcum sunucusu durdu'
            break
        if os.path.exists(f'{CIKTI}/KAPAT'):
            neden = 'Claude isi bitirdi'
            break
        if tun is None or tun.poll() is not None:
            tunel_denemesi += 1
            if tunel_denemesi > 20:
                neden = 'tunel yeniden acilamadi'
                break
            print(time.strftime('%H:%M'), 'tunel kapandi, yeniden aciliyor', flush=True)
            tun, adres = tunel_ac()
            if adres:
                adres_yaz(adres)
            continue
        try:
            d = yerel('/durum?sessiz=1')
        except Exception:
            continue
        bosta = d['bosta_sn'] / 60
        if bosta > BOSTA_KAPAT_DK:
            neden = f'{BOSTA_KAPAT_DK} dakikadir is gelmedi'
            break
        if time.time() - son_yazi > 300:
            son_yazi = time.time()
            calisan = d['calisan']['ad'] if d['calisan'] else '-'
            print(time.strftime('%H:%M'), f"calisan: {calisan} | bekleyen {len(d['bekleyen'])} | biten {len(d['biten'])}"
                  f' | {bosta:.0f} dk bosta', flush=True)
except KeyboardInterrupt:
    neden = 'elle durduruldu'
finally:
    print('\nBitti:', neden, flush=True)
    try:
        for dosya in os.listdir(CIKTI):  # yarim kalan isin ciktisi da Drive a gitsin
            if os.path.isfile(f'{CIKTI}/{dosya}') and dosya != 'KAPAT':
                shutil.copy2(f'{CIKTI}/{dosya}', f'{KOSU}/{dosya}')
        shutil.copy('/content/sunucu.log', f'{KOSU}/sunucu.log')
        with open(f'{OLCUM}/adres.txt', 'w') as f:
            f.write(f'KAPALI\n-\n{DAMGA}\n{int(time.time())}\n{neden}\n')
    except Exception as hata:
        print('Son kopyalama:', hata)
    srv.terminate()
    if tun is not None:
        tun.terminate()
print('Ciktilar:', KOSU)